In [5]:
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece, Unigram, WordLevel
from tokenizers.pre_tokenizers import BertPreTokenizer, ByteLevel
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from pathlib import Path
from typing import List, Literal, Optional
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.normalizers import Sequence, NFKC
from pathlib import Path
from typing import List, Literal, Optional
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from helper_functions import load_wikipedia_text
import numpy as np
from typing import Dict
import tiktoken
from types import SimpleNamespace

/zhome/32/4/214716/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# https://arxiv.org/pdf/2203.15556#page=24.27  in this paper they set the ratio for optimal language model training to 20 tokens per trainable parameter
trainable_params = 2_221_696
TARGET = trainable_params * 20  # target number of characters to load; we fix this for all other experiments

# FIXME: the target does not takeinot account the 70/10/20 split between train/val/test

text_en = load_wikipedia_text("en", TARGET)
text_ru = load_wikipedia_text("ru", TARGET)
text =  text_ru + text_en


In [18]:
import tiktoken
from types import SimpleNamespace
# -------------------------------
#     Pre-trained tokenizer
# -------------------------------

# Load the pre-trained tokenizer
enc = tiktoken.get_encoding("cl100k_base")



class TiktokenWrapper:
    def __init__(self, enc):
        self.encoder = enc

    def encode(self, text: str):
        ids = self.encoder.encode(text)
        tokens = [self.encoder.decode([i]) for i in ids]
        return SimpleNamespace(ids=ids, tokens=tokens)
    
    def decode(self, ids):
        return self.encoder.decode(ids)
    
    def token_to_id(self, token: str):
        # Returns id only if token is exactly one token long
        ids = self.encoder.encode(token)
        if len(ids) == 1:
            return ids[0]
        return None


In [13]:
# --------------------------------------------------
#             Custom tokenizer training
# --------------------------------------------------

TokenizerModel = Literal["bbpe","bpe", "wordpiece", "unigram", "bytelevel"]

def train_custom_tokenizer(
    texts: List[str], 
    model_type: TokenizerModel, 
    vocab_size: int = 16384, 
    min_frequency: int = 2,
    special_tokens: Optional[List[str]] = None,
    save_path: str = f"tokenizers/custom_tokenizer.json",
):

    if special_tokens is None:
        special_tokens = [ "[UNK]", "[EOS]"]

    print(f"Training {model_type.upper()} tokenizer...")
    
    """
    args:
        texts: List of strings to train the tokenizer on
        model_type: Type of tokenizer to train ("bpe", "wordpiece", "unigram", "bytelevel")
        vocab_size: Size of the vocabulary
        min_frequency: Minimum frequency for a token to be included in the vocabulary (only for BPE and WordPiece)
        special_tokens: List of special tokens to include in the vocabulary
        save_path: Path to save the trained tokenizer
    returns:
        Trained tokenizer object (ids and tokens)
    """

    # --------------------------
    # Normal BPE (Unicode tokens)
    # --------------------------
    if model_type == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        tokenizer.normalizer = Sequence([NFKC()])
        tokenizer.pre_tokenizer = BertPreTokenizer()

        trainer = BpeTrainer(
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            special_tokens=special_tokens,
        )

    # --------------------------
    # BYTE-LEVEL BPE  Tokenizer
    # --------------------------
    # https://arxiv.org/pdf/1909.03341 
    
    elif model_type == "bbpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

        # must operate on bytes BEFORE unicode interpretation
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)

        tokenizer.normalizer = None

        ## must be used for proper decoding back into russian characters
        tokenizer.decoder = ByteLevelDecoder()

        # must reconstruct bytes AFTER merging
        tokenizer.post_processor = ByteLevelProcessor()

        trainer = BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=special_tokens
        )
    # --------------------------
    #     Unigram Tokenizer
    # --------------------------
    elif model_type == "unigram":
        tokenizer = Tokenizer(Unigram())
        
        tokenizer.normalizer = Sequence([NFKC()]) 
        tokenizer.pre_tokenizer = BertPreTokenizer()
        
        trainer = UnigramTrainer(
            vocab_size=vocab_size,
            special_tokens=special_tokens,
            unk_token="[UNK]"
        )

    # --------------------------
    #    Wordpiece Tokenizer
    # --------------------------
    elif model_type == "wordpiece":
        tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

        tokenizer.normalizer = Sequence([NFKC()])
        tokenizer.pre_tokenizer = BertPreTokenizer()

        trainer = WordPieceTrainer(
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            special_tokens=special_tokens,
        )
    # --------------------------
    #    Byte-Level Tokenizer
    # --------------------------
    elif model_type == "bytelevel":
        def bytes_to_unicode():
            # we do this to ensure that every byte is mapped, excluding control characters  
            bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
            cs = bs[:]
            n = 0
            for b in range(2**8):
                if b not in bs:
                    bs.append(b)
                    cs.append(2**8 + n)
                    n += 1
            cs = [chr(n) for n in cs]
            return dict(zip(bs, cs))

        byte_map = bytes_to_unicode() 

        vocab: Dict[str, int] = {}

        current_id = 0
        for byte_val in range(256):
            u_char = byte_map[byte_val]
            vocab[u_char] = current_id
            current_id += 1

        
        for token in special_tokens:
            vocab[token] = current_id
            current_id += 1

        
        tokenizer = Tokenizer(BPE(vocab=vocab, merges=[], unk_token="[UNK]"))
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=False) 
        tokenizer.decoder = ByteLevelDecoder()
        tokenizer.normalizer = None

    else:
        raise ValueError(f"Unknown model type: {model_type}")

    if model_type not in ["bytelevel"]:
        tokenizer.train_from_iterator(texts, trainer)

    # Save
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(save_path))

    print(f"{model_type.upper()} tokenizer trained and saved to {save_path}")
    return tokenizer

## Tokenizer training

In [19]:
from pathlib import Path
from tokenizers import Tokenizer

folder_name = "tokenizers"
tokenizer_files = {
    "bbpe": "bbpe_tokenizer.json",
    "bpe": "bpe_tokenizer.json",
    "unigram": "unigram_tokenizer.json",
    "wordpiece": "wordpiece_tokenizer.json",
    "bytelevel": "bytelevel_tokenizer.json",
}

# --------------------------------------------------------
# 1. Check if ALL tokenizer files exist
# --------------------------------------------------------
all_exist = all((Path(folder_name) / fname).exists() 
                for fname in tokenizer_files.values())

tokenizers = {}  # final dict to fill

if all_exist:
    print("All tokenizer files found. Loading tokenizers...")

    for name, fname in tokenizer_files.items():
        tokenizers[name] = Tokenizer.from_file(str(Path(folder_name) / fname))

else:
    print("Some tokenizer files missing. Training tokenizers...")

    # Train each tokenizer
    tokenizers["bbpe"] = train_custom_tokenizer(
        texts=text,
        model_type="bbpe",
        vocab_size=16384,
        save_path=f"{folder_name}/{tokenizer_files['bbpe']}",
    )

    tokenizers["bpe"] = train_custom_tokenizer(
        texts=text,
        model_type="bpe",
        vocab_size=16384,
        save_path=f"{folder_name}/{tokenizer_files['bpe']}",
    )

    tokenizers["unigram"] = train_custom_tokenizer(
        texts=text,
        model_type="unigram",
        vocab_size=16384,
        save_path=f"{folder_name}/{tokenizer_files['unigram']}",
    )

    tokenizers["wordpiece"] = train_custom_tokenizer(
        texts=text,
        model_type="wordpiece",
        vocab_size=16384,
        save_path=f"{folder_name}/{tokenizer_files['wordpiece']}",
    )

    tokenizers["bytelevel"] = train_custom_tokenizer(
        texts=text,
        model_type="bytelevel",
        save_path=f"{folder_name}/{tokenizer_files['bytelevel']}",
    )

# Optional: Load pretrained tokenizer too
c100k_pretrained_tokenizer = TiktokenWrapper(tiktoken.get_encoding("cl100k_base"))
tokenizers["cl100k_base"] = c100k_pretrained_tokenizer


All tokenizer files found. Loading tokenizers...


In [20]:
import json
from pathlib import Path

def save_tiktoken_tokenizer(enc_name: str, save_path: str):
    data = {"tiktoken_encoding": enc_name}

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved TikToken wrapper metadata to {save_path}")


In [21]:
save_tiktoken_tokenizer(
    enc_name="cl100k_base",
    save_path="tokenizers/cl100k_pretrained.json"
)


Saved TikToken wrapper metadata to tokenizers/cl100k_pretrained.json


In [22]:
# --------------------------------------------------
#                 Pre-trained tokenizer
# --------------------------------------------------

# choice of pre-trained tokenizer : "cl100k" "core Language" byte paired encoding (BPE), 100K vocab length
# BPE ensures no UNK, and full coverage of Unicode characters including Cyrillic 

import tiktoken
from types import SimpleNamespace

# -------------------------------
#     Pre-trained tokenizer
# -------------------------------

# Load the pre-trained tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# adding a wrapper to make it compatible with custom tokenizers pipeline
class TiktokenWrapper:
    def __init__(self, enc_name: str):
        self.enc_name = enc_name
        self.encoder = tiktoken.get_encoding(enc_name)

    # --- API: encode returns ids + tokens ---
    def encode(self, text: str):
        ids = self.encoder.encode(text)
        tokens = [self.encoder.decode([i]) for i in ids]
        return SimpleNamespace(ids=ids, tokens=tokens)

    def decode(self, ids):
        return self.encoder.decode(ids)

    def token_to_id(self, token: str):
        ids = self.encoder.encode(token)
        if len(ids) == 1:
            return ids[0]
        return None

    def get_vocab_size(self):
        return self.encoder.n_vocab

    # ---------- saving ----------
    def save(self, path: str):
        data = {
            "type": "tiktoken",
            "encoding_name": self.enc_name
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f"Saved TikToken tokenizer metadata to {path}")

    # ---------- loading ----------
    @staticmethod
    def from_file(path: str):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if data.get("type") != "tiktoken":
            raise ValueError("Not a TikToken tokenizer metadata file")

        return TiktokenWrapper(data["encoding_name"])
    
# creating the wrapped tokenizer
# pretrained_tokenizer = TiktokenWrapper(enc)

# add to tokenizers dict
# -------------------------------
# tokenizers["cl100k_pretrained"] = pretrained_tokenizer

pretrained_tokenizer = TiktokenWrapper("cl100k_base")
pretrained_tokenizer.save("tokenizers/cl100k_pretrained.json")


# sanity check: cl100k_base tokenizer is added to Tokenizer dict
# print("Registered tokenizers:", list(tokenizers.keys()))

# Test on your text
# -------------------------------
tokens = enc.encode("Привет, как дела?")
print(tokens)
print(enc.decode(tokens))

Saved TikToken tokenizer metadata to tokenizers/cl100k_pretrained.json
[54745, 28089, 8341, 11, 52770, 95369, 1506, 30]
Привет, как дела?


## Define metrics


### Tokenizer-only metrics
| Metric                               | Category     | Why tokenizer-only?                                       |
| ------------------------------------ | ------------ | --------------------------------------------------------- |
| **(3) Tokens / Character**           | Efficiency   | Pure tokenization behavior.                               |
| **(4) Average Characters / Token**   | Efficiency   | Depends only on segmentation.                             |
| **(5) Sequence Length Distribution** | Compute cost | Sequence length is produced by tokenizer before training. |
| **(7) OOV Rate**                     | Robustness   | Tokenizer’s ability to cover text.                        |
| **(8) Unicode / Character Coverage** | Robustness   | Tokenizer’s vocabulary coverage of character set.         |
| **(10) Vocab File Size**             | Storage      | Pure tokenizer metadata.                                  |


In [23]:

def tokens_per_character(tokenizer, texts):
    """
    Computes the ratio: total_tokens / total_characters
    Estimate tokenization efficiency --> A low value means the tokenizer is efficient
    """
    total_tokens = 0
    total_chars = 0
    for t in texts:
        enc = tokenizer.encode(t)
        # number of tokens
        total_tokens += len(enc.ids)
        # number of characters
        total_chars += len(t)

    value = total_tokens / max(total_chars, 1)
    print(f"[Tokens/Character] {value:.4f}")
    return value


def avg_characters_per_token(tokenizer, texts):
    """
    Measures how many characters each token represent on average
    Use the *lengths of token strings*, not raw text
    """
    total_chars = 0
    total_tokens = 0

    for t in texts:
        enc = tokenizer.encode(t)
        tokens = enc.tokens
        total_tokens += len(tokens)
        total_chars += sum(len(tok) for tok in tokens)

    value = total_chars / max(total_tokens, 1)
    print(f"[Avg Characters/Token] {value:.4f}")
    return value


# Computes basic statistics on tokenized sequence lengths
def sequence_length_distribution(tokenizer, texts, seq_len=128):
    """
    arg: tokenizer: the tokenizer to analyze
         texts: list of texts to analyze
         seq_len: the sequence length to consider
    returns: numpy array of sequence lengths
    """
    lengths = []

    for t in texts:
        enc = tokenizer.encode(t)
        lengths.append(len(enc.ids))

    lengths = np.array(lengths)

    print(f"[Sequence Length Distribution]")
    print(f"  mean: {lengths.mean():.2f}")
    print(f"  std:  {lengths.std():.2f}")
    print(f"  min:  {lengths.min():.2f}")
    print(f"  max:  {lengths.max():.2f}")
    return lengths

# number of unknown tokens: OOV = Out-of-Vocabulary
def oov_rate(tokenizer, texts):
    unk_id = tokenizer.token_to_id("[UNK]")
    if unk_id is None:
        print("[OOV Rate] this tokenizer has no [UNK] token")
        return 0.0

    total_tokens = 0
    unk_tokens = 0

    for t in texts:
        enc = tokenizer.encode(t)
        ids = enc.ids

        total_tokens += len(ids)
        # count UNK tokens
        unk_tokens += sum(1 for i in ids if i == unk_id)

    value = unk_tokens / max(total_tokens, 1)
    print(f"[OOV Rate] {value:.5f}")
    return value

# Unicode character coverage
def unicode_character_coverage(tokenizer, texts):
    """
    Percentage of UNIQUE characters in the dataset that can be
    encoded without producing UNK tokens
    --> relevant for our multilingual datasets
    """
    all_chars = set("".join(texts))
    unk_id = tokenizer.token_to_id("[UNK]")

    if unk_id is None:
        # No UNK token: by definition, tokenizer can encode everything in some way
        print("[Unicode Coverage] 100.00% (no [UNK] token)")
        return 1.0

    covered_chars = 0

    # Check each character independently
    for ch in all_chars:
        enc = tokenizer.encode(ch)
        ids = enc.ids
        # If encoding the char does NOT produce UNK --> covered
        if unk_id not in ids:
            covered_chars += 1

    value = covered_chars / max(len(all_chars), 1)
    print(f"[Unicode Coverage] {value*100:.2f}%")
    return value


def analyze_tokenizer(tokenizer, texts, verbose=True):
    """
    Compute tokenizer statistics in a single pass for maximum efficiency
    Metrics:
        - tokens_per_character
        - avg_characters_per_token
        - sequence_length_distribution (min/mean/max/std)
        - OOV rate
        - unicode character coverage

    Arg: tokenizer: the tokenizer to analyze
         texts: list of texts to analyze
         verbose: whether to print the results
    returns: dictionary with all metrics
    """

    unk_id = tokenizer.token_to_id("[UNK]")

    # Cumulative statistics
    total_text_chars = 0      # sum of len(text)
    total_tokens = 0          # sum of all token counts
    total_token_chars = 0     # sum of lengths of token strings
    total_unk = 0             # how many tokens became [UNK]
    seq_lengths = []          # list of sequence lengths

    all_chars = set("".join(texts))

    for t in texts:
        # count raw characters
        total_text_chars += len(t)

        # tokenize text
        enc = tokenizer.encode(t)
        ids = enc.ids
        tokens = enc.tokens
        seq_len = len(ids)

        total_tokens += seq_len
        seq_lengths.append(seq_len)

        # accumulate token-string lengths
        total_token_chars += sum(len(tok) for tok in tokens)

        # Count UNK
        if unk_id is not None:
            total_unk += ids.count(unk_id)

    # ratio Tokens/Characters (using raw text chars)
    tokens_per_character = (total_tokens / total_text_chars) if total_text_chars > 0 else 0.0
    # average characters per token (using token string lengths)
    avg_chars_per_token = (total_token_chars / total_tokens) if total_tokens > 0 else 0.0
    seq_lengths = np.array(seq_lengths) if seq_lengths else np.array([])

    # OOV rate --> fraction of tokens that are UNK
    oov = (total_unk / total_tokens) if total_tokens > 0 else 0.0

    # Unicode coverage: per-character test (same logic as unicode_character_coverage)
    if unk_id is None:
        unicode_coverage = 1.0
    else:
        covered_chars = 0
        for ch in all_chars:
            enc_ch = tokenizer.encode(ch)
            if unk_id not in enc_ch.ids:
                covered_chars += 1
        unicode_coverage = covered_chars / max(len(all_chars), 1)

    if verbose:
        print("\n=== Tokenizer Analysis ===")
        print(f"[Tokens/Character]          {tokens_per_character:.4f}")
        print(f"[Avg Characters/Token]      {avg_chars_per_token:.4f}")
        if seq_lengths.size > 0:
            print(f"[Sequence Length Mean]      {seq_lengths.mean():.2f}")
            print(f"[Sequence Length Std]       {seq_lengths.std():.2f}")
            print(f"[Sequence Length Min]       {seq_lengths.min():.2f}")
            print(f"[Sequence Length Max]       {seq_lengths.max():.2f}")
        else:
            print("[Sequence Length]           no data")
        print(f"[OOV Rate]                  {oov:.5f}")
        print(f"[Unicode Coverage]          {unicode_coverage*100:.2f}%")
        print("===========================\n")

    return {
        "tokens_per_character": tokens_per_character,
        "avg_characters_per_token": avg_chars_per_token,
        "sequence_lengths": seq_lengths,
        "sequence_length_mean": float(seq_lengths.mean()) if seq_lengths.size > 0 else 0.0,
        "sequence_length_std": float(seq_lengths.std()) if seq_lengths.size > 0 else 0.0,
        "sequence_length_min": float(seq_lengths.min()) if seq_lengths.size > 0 else 0.0,
        "sequence_length_max": float(seq_lengths.max()) if seq_lengths.size > 0 else 0.0,
        "oov_rate": oov,
        "unicode_coverage": unicode_coverage,
    }

## Simple Encoding Example and Evaluation

In [24]:
# test run:
test_string = "Hello world! Привет, мир! ¿Áéñç𝄢👾👩‍💻👨‍👩‍👧‍👦🏳️‍⚧️\u202E\u0301U\u00A0\uFEFF"
print("--- Test Encodings with Sample String ---")
print(f"Test String: {test_string}")

# Encode and Decode test string
for name, tokenizer in tokenizers.items():
    enc = tokenizer.encode(test_string).ids
    dec = tokenizer.decode(enc)
    print(f"{name}: {dec}")
print()

# Evalate all our tokenizers
for name, tokenizer in tokenizers.items():
    print(f"--- Evaluating {name.upper()} Tokenizer ---")
    analyze_tokenizer(tokenizer, text)

--- Test Encodings with Sample String ---
Test String: Hello world! Привет, мир! ¿Áéñç𝄢👾👩‍💻👨‍👩‍👧‍👦🏳️‍⚧️‮́U ﻿
bbpe:  Hello world! Привет, мир! ¿Áéñç𝄢👾👩‍💻👨‍👩‍👧‍👦🏳️‍⚧️‮́U ﻿
bpe: Hell o world ! При вет , мир ! ¿ Á é ñ ç ‍ ‍ ‍ ‍ ‍ ́ U ﻿
unigram: Hell o world ! При в ет , мир ! ¿ Á é ñ ç ‍ ‍ ‍ ‍ ‍ ́ U
wordpiece: Hell ##o world ! При ##вет , мир ! ¿
bytelevel: Hello world! Привет, мир! ¿Áéñç𝄢👾👩‍💻👨‍👩‍👧‍👦🏳️‍⚧️‮́U ﻿
cl100k_base: Hello world! Привет, мир! ¿Áéñç𝄢👾👩‍💻👨‍👩‍👧‍👦🏳️‍⚧️‮́U ﻿

--- Evaluating BBPE Tokenizer ---

=== Tokenizer Analysis ===
[Tokens/Character]          0.3019
[Avg Characters/Token]      4.5995
[Sequence Length Mean]      881.83
[Sequence Length Std]       1895.42
[Sequence Length Min]       5.00
[Sequence Length Max]       108698.00
[OOV Rate]                  0.00000
[Unicode Coverage]          99.48%

--- Evaluating BPE Tokenizer ---

=== Tokenizer Analysis ===
[Tokens/Character]          0.2772
[Avg Characters/Token]      3.0486
[Sequence Length Mean]      809.74
[Sequence 